In [124]:
import sys
sys.path.insert(0, '..')
from panel_exp.panel_data import long_df_to_paneldataset, PanelDataset, TimePeriod
from panel_exp.design import CompleteRandomization, ThinningDesign, Rerandomization, QuickBlock
from panel_exp.design.design_metrics import imbalance
from panel_exp.methods.scm import SyntheticControl
import cvxpy as cp

In [140]:
w.values

AttributeError: 'Variable' object has no attribute 'values'

In [125]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

long_df = pd.read_csv('smoking.csv')
long_df.columns = ['unnamed', 'unit', 'time_unit', 'y',  'lnincome','beer','age15to24','retprice']


In [127]:
def run(pd):
    control, test = pd.split_control_test_units( pd.treated_units ) 
    control=control.T.values
    test = test.values
    w = cp.Variable((control.shape[1], test.shape[0]))
    constraints = [cp.sum(w) == 1, w >= 0]
    objective = cp.Minimize(cp.sum_squares(test.T - control @ w))
    problem = cp.Problem(objective, constraints)
    problem.solve(verbose=False)
    print(problem.value)

In [128]:
pds = long_df_to_paneldataset(long_df, "time_unit", "unit", "y", ["California"], 1989)
print(pds.summarize())


        Panel Dataset Summary
        ---------------------
        Number of time points: 31
        Number of units: 39
        Number of treated units: 1
        Treated units: ['California']
        Treated periods: [TimePeriod(start=1989, end=None)]
        


In [129]:
wide = pd.pivot_table(long_df, index = 'unit', columns='time_unit' , values='y')
pds2 = PanelDataset(wide, TimePeriod(1989, None), ['California'])

In [130]:

pds3 = PanelDataset(pds.wide_data, TimePeriod(1989,None), ['California']) 
pds4 = PanelDataset(pds.wide_data.copy(), TimePeriod(1989,None), ['California']) 
pds5 = PanelDataset(pds.wide_data*1.1, TimePeriod(1989,None), ['California']) 


In [131]:
def balance_objective(x):
    x = x.reshape((control.shape[1], test.shape[0]))
    # minimize the sum of squared errors
    # subject to an entropy penalty
    imbalance = np.sum(np.square(test.T - control @ x))
    if penalty == "entropy":
        print('here: entropy')
        imbalance += penalty_strength * -np.sum(x * np.log(x))
    elif penalty == "l1":
        print('here: l1')
        imbalance += penalty_strength * np.sum(np.abs(x))
    elif penalty == "l2":
        imbalance += penalty_strength * np.sum(np.square(x))
        print('here: l2', imbalance, x.sum())
    else:
        raise NotImplemented(f"Unknown penalty {penalty}")
    return imbalance

# PDS 1 

In [132]:
control, test = pds.split_control_test_units(pds.treated_units)

import warnings
from scipy.optimize import minimize
penalty_strength=.01
penalty='l2'

control = control.values.T
test = test.values
print(control.nbytes)
x0 = (np.ones((control.shape[1], test.shape[0])) / control.shape[1]).flatten()

simplex_bounds = [(0, 1) for _ in range((control.shape[1] * test.shape[0]))]

res = minimize(
    balance_objective,
    x0,
    tol=.00001,
    method="SLSQP",
    bounds=simplex_bounds,
    constraints={'type':'eq', 'fun': lambda x: np.sum(x.reshape(control.shape[1], test.shape[0]), axis=0) - 1}
)
print(res)

5776
here: l2 4890.723717779711 0.9999999999999998
here: l2 4890.724660761413 1.000000014901161
here: l2 4890.724723960601 1.000000014901161
here: l2 4890.724710403088 1.000000014901161
here: l2 4890.7246341559 1.000000014901161
here: l2 4890.724922564783 1.000000014901161
here: l2 4890.724760402634 1.000000014901161
here: l2 4890.724609273745 1.000000014901161
here: l2 4890.72474249975 1.000000014901161
here: l2 4890.724908309215 1.000000014901161
here: l2 4890.724656067361 1.000000014901161
here: l2 4890.724696481697 1.000000014901161
here: l2 4890.7253456429335 1.000000014901161
here: l2 4890.724788748642 1.000000014901161
here: l2 4890.72481572354 1.000000014901161
here: l2 4890.724629416794 1.000000014901161
here: l2 4890.724684565037 1.000000014901161
here: l2 4890.724814358523 1.000000014901161
here: l2 4890.724633438998 1.000000014901161
here: l2 4890.724610993073 1.000000014901161
here: l2 4890.725076537118 1.000000014901161
here: l2 4890.725628114595 1.000000014901161
here: l

In [146]:
control, test = pds.split_control_test_units( pds.treated_units ) 
control=control.T.values
test = test.values
w = cp.Variable((control.shape[1], test.shape[0]))
constraints = [cp.sum(w) == 1, w >= 0]
objective = cp.Minimize(cp.sum_squares(test.T - control @ w))
problem = cp.Problem(objective, constraints)
problem.solve(verbose=False)
print(problem.value)

52.12957126425191


In [143]:
-np.sum(w.value * np.log(w.value))

/var/folders/cd/dfrqgp4s4ll55cwb7rtgccbw0000gq/T/ipykernel_60266/133296266.py:1: RuntimeWarning: invalid value encountered in log
  -np.sum(w.value * np.log(w.value))


nan

In [151]:
w.value

array([[-4.99047844e-15],
       [-4.23093187e-15],
       [ 1.48107704e-02],
       [ 1.09089624e-01],
       [-5.05218353e-16],
       [-3.37600361e-15],
       [-1.17747966e-15],
       [-1.18187857e-15],
       [-2.83761007e-15],
       [-2.25071924e-15],
       [-2.51888051e-15],
       [-4.60323016e-15],
       [-3.09920155e-15],
       [-1.74892852e-15],
       [-1.85018082e-15],
       [-4.12638118e-15],
       [-2.81742034e-15],
       [ 2.31839951e-01],
       [-1.37715082e-15],
       [ 2.04922584e-01],
       [ 4.54290464e-02],
       [-5.76289267e-16],
       [-1.58619317e-15],
       [-3.36775799e-15],
       [-1.83273491e-15],
       [-3.53808138e-15],
       [-2.79579580e-15],
       [-3.08498133e-15],
       [-4.47629836e-15],
       [-3.33839953e-15],
       [-4.27570768e-15],
       [-2.65688879e-15],
       [ 3.93908024e-01],
       [-4.40864910e-15],
       [-3.86049763e-15],
       [-2.18451925e-15],
       [-2.11331210e-15],
       [-2.50686202e-15]])

# PDS 2: Uses Pivot_table

In [116]:
control, test = pds2.split_control_test_units(pds2.treated_units)

import warnings
from scipy.optimize import minimize
penalty_strength=.01
penalty='l2'

control = control.values.T
test = test.values
print(control.nbytes)
x0 = (np.ones((control.shape[1], test.shape[0])) / control.shape[1]).flatten()

simplex_bounds = [(0, 1) for _ in range((control.shape[1] * test.shape[0]))]

res = minimize(
    balance_objective,
    x0,
    tol=.00001,
    method="SLSQP",
    bounds=simplex_bounds,
    #options = {'ftol': 0, 'eps': -.001} , 
    constraints={'type':'eq', 'fun': lambda x: np.sum(x.reshape(control.shape[1], test.shape[0]), axis=0) - 1}
)
print(res)

5776
here: l2 4890.723717779706 0.9999999999999998
here: l2 4890.724660761409 1.000000014901161
here: l2 4890.724723960596 1.000000014901161
here: l2 4890.724710403084 1.000000014901161
here: l2 4890.724634155895 1.000000014901161
here: l2 4890.724922564777 1.000000014901161
here: l2 4890.724760402628 1.000000014901161
here: l2 4890.7246092737405 1.000000014901161
here: l2 4890.724742499746 1.000000014901161
here: l2 4890.72490830921 1.000000014901161
here: l2 4890.724656067356 1.000000014901161
here: l2 4890.724696481691 1.000000014901161
here: l2 4890.725345642928 1.000000014901161
here: l2 4890.724788748637 1.000000014901161
here: l2 4890.724815723534 1.000000014901161
here: l2 4890.72462941679 1.000000014901161
here: l2 4890.724684565033 1.000000014901161
here: l2 4890.7248143585175 1.000000014901161
here: l2 4890.724633438995 1.000000014901161
here: l2 4890.7246109930675 1.000000014901161
here: l2 4890.725076537114 1.000000014901161
here: l2 4890.725628114589 1.000000014901161
her

In [117]:
run(pds2)

52.12957126425178


# PDS 3. Just uses copy of wide data from PDS 1 

In [118]:
control, test = pds3.split_control_test_units(pds3.treated_units)

import warnings
from scipy.optimize import minimize
penalty_strength=.01
penalty='l2'

control = control.values.T
test = test.values

x0 = (np.ones((control.shape[1], test.shape[0])) / control.shape[1]).flatten()

simplex_bounds = [(0, 1) for _ in range((control.shape[1] * test.shape[0]))]

res = minimize(
    balance_objective,
    x0,
    tol=.00001,
    method="SLSQP",
    bounds=simplex_bounds,
    constraints={'type':'eq', 'fun': lambda x: np.sum(x.reshape(control.shape[1], test.shape[0]), axis=0) - 1}
)
print(res)

here: l2 4890.723717779711 0.9999999999999998
here: l2 4890.724660761413 1.000000014901161
here: l2 4890.724723960601 1.000000014901161
here: l2 4890.724710403088 1.000000014901161
here: l2 4890.7246341559 1.000000014901161
here: l2 4890.724922564783 1.000000014901161
here: l2 4890.724760402634 1.000000014901161
here: l2 4890.724609273745 1.000000014901161
here: l2 4890.72474249975 1.000000014901161
here: l2 4890.724908309215 1.000000014901161
here: l2 4890.724656067361 1.000000014901161
here: l2 4890.724696481697 1.000000014901161
here: l2 4890.7253456429335 1.000000014901161
here: l2 4890.724788748642 1.000000014901161
here: l2 4890.72481572354 1.000000014901161
here: l2 4890.724629416794 1.000000014901161
here: l2 4890.724684565037 1.000000014901161
here: l2 4890.724814358523 1.000000014901161
here: l2 4890.724633438998 1.000000014901161
here: l2 4890.724610993073 1.000000014901161
here: l2 4890.725076537118 1.000000014901161
here: l2 4890.725628114595 1.000000014901161
here: l2 489

In [119]:
run(pds3)

52.12957126425191


# PDS 4 . Using .copy()

In [120]:
control, test = pds4.split_control_test_units(pds4.treated_units)

import warnings
from scipy.optimize import minimize
penalty_strength=.01
penalty='l2'

control = control.values.T
test = test.values

x0 = (np.ones((control.shape[1], test.shape[0])) / control.shape[1]).flatten()

simplex_bounds = [(0, 1) for _ in range((control.shape[1] * test.shape[0]))]

res = minimize(
    balance_objective,
    x0,
    tol=1e-16 , 
    method="SLSQP",
    bounds=simplex_bounds,
    constraints={'type':'eq', 'fun': lambda x: np.sum(x.reshape(control.shape[1], test.shape[0]), axis=0) - 1}
)
print(res)

here: l2 4890.723717779706 0.9999999999999998
here: l2 4890.724660761409 1.000000014901161
here: l2 4890.724723960596 1.000000014901161
here: l2 4890.724710403084 1.000000014901161
here: l2 4890.724634155895 1.000000014901161
here: l2 4890.724922564777 1.000000014901161
here: l2 4890.724760402628 1.000000014901161
here: l2 4890.7246092737405 1.000000014901161
here: l2 4890.724742499746 1.000000014901161
here: l2 4890.72490830921 1.000000014901161
here: l2 4890.724656067356 1.000000014901161
here: l2 4890.724696481691 1.000000014901161
here: l2 4890.725345642928 1.000000014901161
here: l2 4890.724788748637 1.000000014901161
here: l2 4890.724815723534 1.000000014901161
here: l2 4890.72462941679 1.000000014901161
here: l2 4890.724684565033 1.000000014901161
here: l2 4890.7248143585175 1.000000014901161
here: l2 4890.724633438995 1.000000014901161
here: l2 4890.7246109930675 1.000000014901161
here: l2 4890.725076537114 1.000000014901161
here: l2 4890.725628114589 1.000000014901161
here: l2

In [121]:
run(pds4)

52.12957126425178


In [46]:
pds3 = PanelDataset(pds.wide_data.copy(), TimePeriod(1989,None), ['California']) 


In [29]:
pds.wide_data[pds.wide_data.columns[-13:]] = pds.wide_data[pds.wide_data.columns[-13:]]*1.1

In [30]:
pds.wide_data[pds.wide_data.columns[-13:]] 

time_unit,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000
unit,,,,,,,,,,,,,
Alabama,123.309998,116.159998,119.459998,118.690002,120.009998,119.350000,117.809998,112.859998,111.540002,115.390002,116.819997,110.769997,105.819997
Arkansas,133.650000,130.130003,124.409998,128.480003,138.600000,125.180003,119.680003,124.300000,121.769997,119.569997,120.450000,115.280003,109.340002
California,99.109998,90.640002,85.580003,75.569997,74.250000,69.740002,64.459998,62.040002,59.950000,59.179999,57.529999,51.920001,45.759998
Colorado,104.059998,97.680003,96.140002,99.219997,97.130003,97.459998,98.009998,93.940002,91.409998,89.430003,89.319997,87.559998,80.300000
Connecticut,115.280003,110.659998,100.650000,95.369997,91.850000,87.009998,84.259998,87.230003,83.600000,83.490002,83.050000,80.740002,78.540002
Delaware,150.810007,144.869997,139.919997,130.680003,132.000000,136.180003,138.709998,139.919997,141.130003,136.509998,146.080003,153.450000,154.769997
Georgia,136.509998,128.809998,125.180003,120.559998,120.119997,120.119997,118.580003,110.330003,112.969997,110.659998,110.550000,106.809998,97.240002
Idaho,92.950000,86.240002,99.109998,93.940002,93.609998,95.369997,102.300000,86.019997,80.959998,82.500000,86.790002,82.609998,73.590002
Illinois,118.359998,115.059998,103.509998,105.709998,104.280003,104.059998,94.269997,92.730003,89.980003,87.559998,88.330003,79.419997,77.000000


In [56]:
a = np.ones(4)

In [57]:
a

array([1., 1., 1., 1.])

In [59]:
b = a.copy()

array([ True,  True,  True,  True])